# Product Text Classification with OpenAI Embeddings

Steps:

- Load and filter dataset

- Create embeddings with text-embedding-3-small.

- Split train/test (stratify by category).

- Train Random Forest classifier.

- Evaluate with accuracy & classification report.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import os
from dotenv import load_dotenv
from pathlib import Path

# Force loading .env from this repo
REPO_DIR = Path(r"c:\Users\maribel.fraire\ML classifier\tp_final")
load_dotenv(REPO_DIR / ".env")

True

In [55]:
api_key=os.getenv("OPENAI_KEY")
client = OpenAI(api_key=api_key)

### Import and filter dataset based on a minimum number of samples per class

In [2]:
df_input = pd.read_csv('train.csv')

In [3]:
df_input = df_input.drop(['language', 'label_quality'], axis=1)
df_input = df_input.rename(columns={'title': 'text'})

# Same technical filter as 02_tp_final_beto_all: MIN_SAMPLES = 20
MIN_SAMPLE = 20
category_counts = df_input['category'].value_counts()
valid_categories = category_counts[category_counts >= MIN_SAMPLE].index
df_filtered = df_input[df_input['category'].isin(valid_categories)]

# Sample 10k (stratify by category)
df_sample, _ = train_test_split(
    df_filtered,
    train_size=10_000,
    random_state=42,
    stratify=df_filtered['category']
)
df_sample = df_sample.reset_index(drop=True)

In [4]:
print(f"Numero de clases: {df_sample['category'].nunique()}")

display(df_sample.head(3))

Numero de clases: 1525


,text,category
0,"Porta Balcão Alumínio Brilhante 2,10 X 1,50 - ...",GARAGE_DOORS
1,Cama Nido Mas Alta Doble En Madera Maciza. Fab...,BEDS
2,Batedor Milk Shake E Multiprocessador Sd 2015 ...,MIXERS


Build the Batch API input file (JSONL)

In [18]:
import json
# Build JSONL for Batch API (one request per text)
# Max 50,000 inputs per batch for /v1/embeddings; 10k is fine.
batch_file_path = "batch_embeddings_10k.jsonl"

with open(batch_file_path, 'w', encoding='utf-8') as f:
    for idx, row in df_sample.iterrows():
        task = {
            "custom_id": str(idx),
            "method": "POST",
            "url": "/v1/embeddings",
            "body": {
                "model": "text-embedding-3-small",
                "input": row['text'][:8191]  # model limit; truncate if needed
            }
        }
        f.write(json.dumps(task, ensure_ascii=False) + '\n')

In [19]:
with open(batch_file_path, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i >= 3:
            break
        obj = json.loads(line)
        print(json.dumps(obj, indent=4, ensure_ascii=False))


{
    "custom_id": "0",
    "method": "POST",
    "url": "/v1/embeddings",
    "body": {
        "model": "text-embedding-3-small",
        "input": "Porta Balcão Alumínio Brilhante 2,10 X 1,50 - 3 Folhas"
    }
}
{
    "custom_id": "1",
    "method": "POST",
    "url": "/v1/embeddings",
    "body": {
        "model": "text-embedding-3-small",
        "input": "Cama Nido Mas Alta Doble En Madera Maciza. Fabricantes. ."
    }
}
{
    "custom_id": "2",
    "method": "POST",
    "url": "/v1/embeddings",
    "body": {
        "model": "text-embedding-3-small",
        "input": "Batedor Milk Shake E Multiprocessador Sd 2015 1200 Watts"
    }
}


Upload file and create batch job

In [58]:
# Upload input file
with open(batch_file_path, 'rb') as f:
    batch_file = client.files.create(file=f, purpose="batch")

# Create batch job (50% discount, completion within 24h)
batch_job = client.batches.create(
    input_file_id=batch_file.id,
    endpoint="/v1/embeddings",
    completion_window="24h"
)

print(f"Batch ID: {batch_job.id}")
print(f"Status: {batch_job.status}")

Batch ID: batch_698754dda0ac8190a2beec3d23190ac1
Status: validating


Check status

Batches puede tardar 24 hs

In [62]:
# Poll until completed (or check later)
import datetime


batch_job = client.batches.retrieve(batch_job.id)
print(datetime.datetime.now())
print(batch_job.status)  # validating -> in_progress -> completed (or failed)
# batch_job.request_counts  # total, completed, failed

2026-02-07 12:11:59.167032
in_progress


In [100]:
# Poll until completed (or check later)
import datetime


batch_job = client.batches.retrieve(batch_job.id)
print(datetime.datetime.now())
print(batch_job.status)  # validating -> in_progress -> completed (or failed)
# batch_job.request_counts  # total, completed, failed

2026-02-07 14:50:59.446428
completed


In [101]:
batch_job.request_counts

BatchRequestCounts(completed=10000, failed=0, total=10000)

In [97]:
batch_job.error_file_id

Download results and attach embeddings to the 10k sample

In [102]:
# After status == "completed"
result_content = client.files.content(batch_job.output_file_id).content

# Save locally
with open("batch_embeddings_10k_results.jsonl", 'wb') as f:
    f.write(result_content)

# Parse and merge back to df_sample by custom_id
# body["data"] ensures non-empty list (avoids IndexError);
results_by_id = {}
with open("batch_embeddings_10k_results.jsonl", 'r', encoding='utf-8') as f:
    for line in f:
        obj = json.loads(line)
        cid = obj["custom_id"]
        body = obj.get("response", {}).get("body", {})
        if "data" in body and body["data"]:
            results_by_id[cid] = body["data"][0]["embedding"]

# Attach to dataframe (custom_id was str(idx))
df_sample['embedding_small'] = df_sample.index.astype(str).map(results_by_id)

# Drop rows that failed (if any)
df_sample = df_sample.dropna(subset=['embedding_small'])

In [105]:
df_sample

,text,category,embedding_small
0,"Porta Balcão Alumínio Brilhante 2,10 X 1,50 - ...",GARAGE_DOORS,"[0.0044723116, 0.016217154, 0.052220736, -0.00..."
1,Cama Nido Mas Alta Doble En Madera Maciza. Fab...,BEDS,"[0.010064922, 0.008933017, 0.05743489, -0.0027..."
2,Batedor Milk Shake E Multiprocessador Sd 2015 ...,MIXERS,"[-0.014096678, -0.009872323, -0.035747223, 0.0..."
3,Calça Motocross Pro Tork Rosa Jett Evolution,MOTORCYCLE_PANTS,"[0.03976407, -0.014263949, -0.011598813, -0.00..."
4,Suporte L.e Morcego Barra Estabiliza Corsa Wag...,TORSION_BARS,"[0.014268805, 0.0017950984, -0.023490064, 0.01..."
...,...,...,...
9995,Processador Intel Core I3-8100 8ªger / 3.6 Ghz...,COMPUTER_PROCESSORS,"[-0.014076135, 0.031874884, 0.00051688286, -0...."
9996,Cargador Original 19.5v 3.9a Sony Vaio Vgn-a11,LAPTOP_CHARGERS,"[0.03520628, -0.04706835, -0.03106933, -0.0161..."
9997,Kit Imprimible Empresarial Colosal Diamante,PARTY_PRINTABLE_KITS,"[0.04663463, 0.008428392, 0.00832938, 0.014616..."
9998,34 Roldanas Duplas Com Freio Para Envidraçamen...,INDUSTRIAL_PULLEYS,"[0.017073616, 0.005825752, 0.030943546, 0.0165..."


### Split with stratification


Para generar los datasets de train, valid y test, el número mínimo de grupos para cualquier clase no puede ser inferior a 2. Sin embargo, la muestra de 10 000 está estratificada, pero con muchas categorías, algunas pueden terminar con solo una fila en esa muestra. Para evitar este ValueError se agrega la validacion vc >= 2, asegurando así una mínima muestra por categoria o clase.

In [134]:
# Se requieren suficientes muestras por clase para que el 30% tenga al menos 2
min_per_class = 2
vc = df_sample['category'].value_counts()
df_ok = df_sample[df_sample['category'].isin(vc[vc >= min_per_class].index)]

# First split: 70% train, 30% temp (test)
X_train, X_test, y_train, y_test = train_test_split(
    df_ok['embedding_small'], df_ok['category'],
    test_size=0.3, random_state=42, stratify=df_ok['category']
)

In [135]:
len(X_train), len(X_test), len(y_train), len(y_test)

(6880, 2949, 6880, 2949)

In [136]:
X_train.shape, y_train.shape, X_test.shape, y_test.shape

((6880,), (6880,), (2949,), (2949,))

In [137]:
X_train.head()

2363    [-0.03355232, -0.019283365, -0.020930255, 0.00...
2105    [0.056291323, 0.0037901888, 0.017336605, 0.019...
3584    [-0.007987967, 0.018646989, -0.023850506, -0.0...
1431    [-3.867397e-05, 0.012270894, -0.048396513, -0....
1425    [0.019588238, 0.0389153, -0.028658185, 0.00171...
Name: embedding_small, dtype: object

scikit-learn
fit(X, y, sample_weight=None)

X_train is a serie of list, but RandomForestClassifier needs 2D array
X: {array-like, sparse matrix} of shape (`n_samples`, `n_features`)

In [138]:
# convert serie to list
import numpy as np
X_train = np.array(X_train.tolist())
X_test  = np.array(X_test.tolist())

len(df_ok), X_train.shape, X_test.shape

(9829, (6880, 1536), (2949, 1536))

### Model and Train: RandomForestClassifier

In [139]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

clf = RandomForestClassifier(n_estimators=200)
clf.fit(X_train, y_train)
pred_rfc = clf.predict(X_test)

#### KNeighborsClassifier

Data preparation for Knn:

- Rescale Data : KNN performs better when data is on the same scale. Normalizing to [0, 1] or standardizing (for Gaussian distributions) improves accuracy.

- Reduce Dimensionality : KNN works best with fewer features. In high-dimensional data, feature selection can improve performance by reducing irrelevant variables. [link](https://medium.com/@RobuRishabh/knn-k-nearest-neighbour-5ae18ae8e274)

In [140]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [141]:
from sklearn.decomposition import PCA

pca = PCA(n_components=50)
X_train_reduced = pca.fit_transform(X_train)
X_test_reduced = pca.transform(X_test)

In [142]:
X_train.shape, X_test.shape

((6880, 1536), (2949, 1536))

In [143]:
X_train_reduced.shape, X_test_reduced.shape

((6880, 50), (2949, 50))

In [144]:
from sklearn.neighbors import KNeighborsClassifier

knn_clf = KNeighborsClassifier(n_neighbors=5)
knn_clf.fit(X_train, y_train)
pred_knn = knn_clf.predict(X_test)

In [145]:
knn_clf_scaled = KNeighborsClassifier(n_neighbors=5)
knn_clf_scaled.fit(X_train_scaled, y_train)
pred_knn_scaled = knn_clf_scaled.predict(X_test_scaled)

In [146]:
knn_clf_reduced = KNeighborsClassifier(n_neighbors=5)
knn_clf_reduced.fit(X_train_reduced,y_train)
pred_knn_reduced = knn_clf_reduced.predict(X_test_reduced)

In [147]:
print(f'Random Forest acc: {accuracy_score(y_test, pred_rfc)}')
print(f'KNN acc: {accuracy_score(y_test,pred_knn)}')
print(f'KNNScaled acc: {accuracy_score(y_test,pred_knn_scaled)}')
print(f'KNNReduced acc {accuracy_score(y_test, pred_knn_reduced)}')
# print(f'Ensamble acc: {accuracy_score(y_test,ensamble_pred_2)}')

Random Forest acc: 0.19057307561885384
KNN acc: 0.4818582570362835
KNNScaled acc: 0.47677178704645645
KNNReduced acc 0.37843336724313326
